In [ ]:
# vehicle Detection - Preparing data

In [ ]:
from pathlib import Path

dataset = Path(r"C:\Users\Student\Downloads\26082217\TRANSSET_v2\TRANSSET_v2")

images = dataset / "images"
labels = dataset / "label_txt"

print("Images:", len(list(images.glob("*.jpg"))))
print("Labels:", len(list(labels.glob("file_*.txt"))))

In [ ]:
image_names = {f.stem for f in images.glob("*.jpg")}
label_names = {f.stem for f in labels.glob("file_*.txt")}

print("Images without lables:", len(image_names - label_names))
print("Lables without lables:", len(label_names - image_names))

In [ ]:
from collections import Counter

vehicle_counts = Counter()

for file in labels.glob("file_*.txt"):
    for line in file.read_text().splitlines():
        if line.strip():
            category = int(line.split()[0])
            vehicle_counts[category] += 1

classes = ["car", "hatchback", "pickup truck", "sedan", "suv", "trailer", "truck", "van"]

for i, name in enumerate(classes):
    print(name, ":", vehicle_counts[i])

In [ ]:
errors = 0

for file in labels.glob("file_*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()

        try:
            if len(parts) != 5:
                errors += 1
                continue

            category = int(parts[0])
            x, y, w, h = map(float, parts[1:])

            if not (
                0 <= category < 8
                and 0 <= x <= 1
                and 0 <= y <= 1
                and 0 < w <= 1
                and 0 < h <= 1
                and x - w / 2 >= 0
                and x + w / 2 <= 1
                and y - h / 2 >= 0
                and y + h / 2 <= 1
            ):
                errors += 1

        except ValueError:
            errors += 1

print("Invalid annotations:", errors)

In [ ]:
from collections import Counter

problems = Counter()

for file in labels.glob("file_*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()

        if len(parts) != 5:
            problems["Wrong number of values"] += 1
            continue

        try:
            category = int(parts[0])
            x, y, w, h = map(float, parts[1:])

            if not 0 <= category < 8:
                problems["Invalid category"] += 1

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                problems["Invalid coordinates"] += 1

            elif x - w / 2 < 0 or x + w / 2 > 1 or y - h / 2 < 0 or y + h / 2 > 1:
                problems["Box outside image"] += 1

        except ValueError:
            problems["Invalid numbers"] += 1

print(problems)

In [ ]:
shown = 0

for file in labels.glob("file_*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()

        if len(parts) == 5:
            category = int(parts[0])
            x, y, w, h = map(float, parts[1:])

            if x - w / 2 < 0 or x + w / 2 > 1 or y - h / 2 < 0 or y + h / 2 > 1:
                print(file.name, ":", line)
                shown += 1

            if shown == 5:
                break

    if shown == 5:
        break

In [ ]:
for file in labels.glob("file_*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()

        if len(parts) == 5:
            x, y, w, h = map(float, parts[1:])

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                print(file.name, ":", line)

In [ ]:
clean_labels = dataset / "clean_labels"
clean_labels.mkdir(exist_ok=True)

for file in labels.glob("file_*.txt"):
    valid_lines = []

    for line in file.read_text().splitlines():
        parts = line.split()

        if len(parts) != 5:
            continue

        category = int(parts[0])
        x, y, w, h = map(float, parts[1:])

        if w <= 0 or h <= 0:
            continue

        x1 = max(0, x - w / 2)
        y1 = max(0, y - h / 2)
        x2 = min(1, x + w / 2)
        y2 = min(1, y + h / 2)

        if x2 <= x1 or y2 <= y1:
            continue

        x = (x1 + x2) / 2
        y = (y1 + y2) / 2
        w = x2 - x1
        h = y2 - y1

        valid_lines.append(f"{category} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    (clean_labels / file.name).write_text("\n".join(valid_lines))

print("Cleaned label files:", len(list(clean_labels.glob("*.txt"))))

In [ ]:
errors = 0

for file in clean_labels.glob("*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            errors += 1
            continue

        category = int(parts[0])
        x, y, w, h = map(float, parts[1:])

        if not (
            0 <= category < 8
            and x - w / 2 >= 0
            and x + w / 2 <= 1.000001
            and y - h / 2 >= 0
            and y + h / 2 <= 1.000001
            and w > 0
            and h > 0
        ):
            errors += 1

print("Invalid cleaned annotations:", errors)

In [ ]:
shown = 0

for file in clean_labels.glob("*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()
        x, y, w, h = map(float, parts[1:])

        if (
            x - w / 2 < 0
            or x + w / 2 > 1.000001
            or y - h / 2 < 0
            or y + h / 2 > 1.000001
            or w <= 0
            or h <= 0
        ):
            print(file.name, ":", line)
            shown += 1

        if shown == 5:
            break

    if shown == 5:
        break

In [ ]:
small_errors = 0
large_errors = 0

for file in clean_labels.glob("*.txt"):
    for line in file.read_text().splitlines():
        parts = line.split()
        x, y, w, h = map(float, parts[1:])

        outside = max(0, -(x - w / 2), x + w / 2 - 1, -(y - h / 2), y + h / 2 - 1)

        if outside > 0:
            if outside <= 0.00001:
                small_errors += 1
            else:
                large_errors += 1

print("Small rounding errors:", small_errors)
print("Larger errors:", large_errors)

In [ ]:
from PIL import Image

bad_images = 0

for file in images.glob("*.jpg"):
    try:
        with Image.open(file) as img:
            img.verify()
    except Exception:
        bad_images += 1

print("Damaged images:", bad_images)

In [ ]:
from sklearn.model_selection import train_test_split

image_files = sorted(images.glob("*.jpg"))

train, remaining = train_test_split(image_files, test_size=0.2, random_state=42)

val, test = train_test_split(remaining, test_size=0.5, random_state=42)

print("Training:", len(train))
print("Validation:", len(val))
print("Testing:", len(test))

In [ ]:
import shutil

clean_labels = dataset / "clean_labels"
output = dataset.parent / "TRANSSET_split"

for name, files in [("train", train), ("val", val), ("test", test)]:
    image_folder = output / name / "images"
    label_folder = output / name / "labels"

    image_folder.mkdir(parents=True, exist_ok=True)
    label_folder.mkdir(parents=True, exist_ok=True)

    for image in files:
        shutil.copy2(image, image_folder / image.name)

        label = clean_labels / (image.stem + ".txt")
        shutil.copy2(label, label_folder / label.name)

    print(name, ":", len(files), "images saved")

In [ ]:
for name in ["train", "val", "test"]:
    image_folder = output / name / "images"
    label_folder = output / name / "labels"

    image_names = {f.stem for f in image_folder.glob("*.jpg")}
    label_names = {f.stem for f in label_folder.glob("*.txt")}

    print(name)
    print("Images:", len(image_names))
    print("Labels:", len(label_names))
    print("Missing labels:", len(image_names - label_names))
    print("Missing images:", len(label_names - image_names))
    print()

In [ ]:
train_names = {f.name for f in train}
val_names = {f.name for f in val}
test_names = {f.name for f in test}

duplicates = (train_names & val_names) | (train_names & test_names) | (val_names & test_names)

print("Images in multiple groups:", len(duplicates))
print("Total unique images:", len(train_names | val_names | test_names))